# Charging trace player

Run all cells, then use **Previous frame / Next frame** to step through the
selected trace, or **Play / Pause** to animate it. The slider jumps to any step;
the other controls change speed, jump to the ends, or enable looping.

Choose a name from `TRACES` in the last cell and rerun that cell to
switch traces. These are illustrative, predefined scene snapshots. Each frame
updates the cars, station grids, costs, and caption together. Write `9` in
a station grid to show a crossed cell. Each trace frame supplies its own
cross pattern.

The player uses Matplotlib's embedded HTML controls with the normal notebook
backend; no widget extension or video encoder is needed. If saved controls are
inactive, trust the notebook or rerun its cells.

In [41]:
import math

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.transforms import Affine2D

GRAY, BLUE = "#595959", "#4285f4"
COLORS = ["#eeeeee", GRAY, BLUE, "#e6b8af"]
STATIONS = ["ST1", "ST2"]


def plot_step(grids, costs=(1, 2), cars=None, *, ax=None, title=None):
    """Draw the exact station grids supplied by a trace frame."""
    if cars is None:
        cars = [(3.25, 0, "A", (1, 0)), (6.5, 0, "B", (1, 0)),
                (0, 2, "C", (2, 0))]
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 6))
    else:
        fig = ax.figure
        ax.clear()
    ax.set(xlim=(-0.5, 8.2), ylim=(-0.5, 4.6), aspect="equal")
    ax.axis("off")
    if title is not None:
        ax.set_title(title, color=GRAY, fontsize=12, pad=12)

    def edge(start, end, label, arrow=True):
        ax.annotate("", end, start, arrowprops=dict(
            arrowstyle="-|>" if arrow else "-", color=GRAY, lw=1.2))
        x, y = [(a + b) / 2 for a, b in zip(start, end)]
        ax.text(x + 0.12, y + 0.12, str(label), color=GRAY, fontsize=11)

    for left, right in [(0.55, 2.7), (3.85, 5.95), (7.1, 8)]:
        edge((left, 2), (right, 2), 1, arrow=False)

    for name, x, cost in zip(STATIONS, (3.25, 6.5), costs):
        grid = grids[name]
        edge((x, 0.5), (x, 1.65), cost)
        edge((x, 2.85), (x, 4.05), 1)
        ax.plot(x, 4.25, marker="*", ms=18, mfc="white", mec=GRAY, mew=1.2)
        ax.text(x, 2, name, ha="center", va="center", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.6", fc="white", ec=GRAY, lw=1.5))
        ax.plot([x - 0.14, x - 0.21, x - 0.21, x - 0.15],
                [2.08, 2.08, 1.89, 1.89], color=GRAY, lw=1.5)
        for r, row in enumerate(grid):
            for c, value in enumerate(row):
                if value not in (0, 1, 2, 3, 9):
                    raise ValueError("Grid cells must have values 0, 1, 2, 3, or 9.")
                left = x + (c - len(row) / 2) * 0.13
                bottom = 2.55 - r * 0.16
                ax.add_patch(Rectangle((left, bottom), 0.13, 0.16,
                                       facecolor=COLORS[0 if value == 9 else value],
                                       edgecolor=GRAY, lw=1.5))
                if value == 9:
                    for ys in ((bottom + 0.02, bottom + 0.14),
                               (bottom + 0.14, bottom + 0.02)):
                        ax.plot((left + 0.015, left + 0.115), ys,
                                color="#c62828", lw=1.3, zorder=3)

    for x, y, label, cells in cars:
        if len(cells) != 2 or any(cell not in range(len(COLORS)) for cell in cells):
            raise ValueError("Each car needs two grid cells with values 0..3.")
        t = Affine2D().rotate_deg(90 if label == "C" else 0).translate(x, y) + ax.transData
        ax.add_patch(FancyBboxPatch((-0.055, -0.11), 0.11, 0.22,
                                   boxstyle="round,pad=0.015", color="black", transform=t))
        for wy in (-0.075, 0.04):
            ax.add_patch(Rectangle((-0.045, wy), 0.09, 0.04, color="white", transform=t))
        dx, dy = (0, -0.24) if label == "C" else (0.24, 0)
        for slot, value in enumerate(cells):
            ax.add_patch(Rectangle((x + dx - 0.13 + slot * 0.13,
                                    y + dy - 0.08), 0.13, 0.16,
                                   facecolor=COLORS[value], edgecolor=GRAY,
                                   lw=1.2))
        ax.text(x - 0.3, y + 0.17, label, color=GRAY, fontsize=12)

    fig.tight_layout()
    return fig, ax

## Predefined traces

Add or edit a named list below to define a trace. Every item is one complete
snapshot passed to `plot_step`:

- `grids`: station grids keyed by `"ST1"` and `"ST2"`, with rows ordered
  from top to bottom. Cells use `0` = empty, `1` = gray, `2` = blue,
  `3` = pink, and `9` = crossed. Put each `9` directly in the frame.
  For example, `[[9, 9, 0], [0, 0, 0]]` crosses the first two cells
  in the top row. You can place a `9` anywhere in either row.
- `cars`: a list of `(x, y, label, cells)` tuples. `cells` is exactly two
  values from `0` to `3`, such as `(2, 0)` for a blue and empty cell beside C.
  Each frame specifies these cells alongside the car position. An empty list
  hides all cars; omitting it uses the original A/B/C positions and cell pairs.
- `costs`: the two station costs (defaults to `(1, 2)`).
- `title`: an optional caption shown alongside the step number.
- `time`: optional label for the current frame. Playback shows the
  zero-based frame index when omitted; this label does not change grids.

Frames are independent: their grids and crosses come from that frame.
Stepping backward shows the earlier grid exactly as specified. Playback
spends the same time on every snapshot.

## Scenario: contested chargers

`"Contested chargers"` is a six-step trace where the two stations are the scarce
resource. Travel costs are the ones drawn on the map: A needs 1 step to reach
ST1, B needs 2 steps to reach ST2, C crosses the corridor west to east at 1 step
per leg. Every vehicle starts at half battery and needs exactly one charging
step; a charger serves one vehicle per step.

| t | A (gray) | B (pink) | C (blue) |
| --- | --- | --- | --- |
| 0 | departs, 1 step to ST1 | departs, 2 steps to ST2 | enters west, 1 step to ST1 |
| 1 | **charges at ST1** | in transit | reaches ST1, **refused** |
| 2 | full, climbs north | reaches ST2, **charges** | reaches ST2, **refused** |
| 3 | at its goal | full, climbs north | **charges at ST2** |
| 4 | | at its goal | full, drives east |
| 5 | | | exits east |

The two conflicts are causally linked: A wins step 1 at ST1, so C drives on
instead of queueing, which puts it at ST2 at exactly the step B claims. C is
refused twice and finishes its trip two steps later than a free run would take.

In the station grids each column is one time step (`t = 0..4`), the top row
shows who holds the charger then (`9` = step spent with nobody plugged in), and
the bottom row keeps a record of the requests that were refused. The two cells
beside a car are its battery.

In [42]:
INITIAL_GRIDS = [
    [[3, 0, 1, 2, 0], [0, 0, 0, 0, 0]],
    [[3, 0, 0, 1, 1], [0, 0, 0, 0, 0]],
]
STATION_2_GRIDS = [
    [[3, 0, 1, 0, 0], [0, 0, 0, 0, 0]],
    [[3, 0, 0, 1, 1], [0, 2, 0, 0, 0]],
]
FINISHED_GRIDS = [[[0] * 5 for _ in range(2)] for _ in range(2)]

G = 1  # gray  - vehicle A
B = 2  # blue  - vehicle C
P = 3  # pink  - vehicle B
X = 9  # step that elapsed with nobody plugged in

# Scenario B reads the station grids as booking sheets: column j is the step
# from t = j to t = j + 1, the top row says which vehicle has charging access
# during that step, and the bottom row records a request that was refused
# because the step was already taken. A charging vehicle holds its charger for
# a whole column, so it stands at the station in two consecutive frames.
#
# The two cells beside a vehicle are its fuel, so a tank holds 2 units. Fuel
# drops by the cost drawn on the road as the vehicle covers it - B is one unit
# down halfway along its cost-2 approach to ST2 - and a charging step refills
# the tank. Every vehicle is therefore forced to charge exactly once:
#
#   A  1 unit,  start -1-> ST1 (empty) -charge-> -1-> goal (1 left)
#   B  2 units, start -2-> ST2 (empty) -charge-> -1-> goal (1 left)
#   C  2 units, west -1-> ST1 (1 left) -charge-> -1-> ST2 -1-> east (empty)
#
# The booking sheet is agreed before anyone moves, and one booking propagates
# into the next: B owns ST2 for t = 2-3, which is exactly the step C would need
# there (driving straight past ST1 would leave C at ST2 with an empty tank), so
# C spends its ST1 stop at t = 1-2 instead. That in turn leaves A - which rolls
# up to ST1 empty at t = 1 - with the later, more inconvenient ST1 step
# t = 2-3. A and C both finish one step later than an empty network allows.

TRACES = {
    "Scenario A": [
        {"grids": {"ST1": [[0, 0, 1, 2, 0], [0, 0, 0, 0, 0]],
                   "ST2": [[0, 0, 0, 1, 1], [0, 0, 0, 0, 0]]}},
        {"grids": {"ST1": [[9, 0, 1, 2, 0], [0, 0, 0, 0, 0]],
                   "ST2": [[9, 0, 0, 1, 1], [0, 0, 0, 0, 0]]},
         "cars": [(3.25, 0.9, "A", (1, 0)),
                  (6.5, 0.9, "B", (1, 0)),
                  (1.6, 2, "C", (2, 0))]},
        {"grids": {"ST1": [[9, 9, 1, 2, 0], [0, 0, 0, 0, 0]],
                   "ST2": [[9, 9, 0, 1, 1], [0, 0, 0, 0, 0]]},
         "cars": [(3.25, 3.4, "A", (1, 0)),
                  (6.5, 3.4, "B", (1, 0)),
                  (2.65, 2, "C", (2, 0))]},
        {"grids": {"ST1": [[9, 9, 9, 2, 0], [0, 0, 0, 0, 0]],
                   "ST2": [[9, 9, 9, 0, 0], [0, 0, 0, 0, 0]]},
         "cars": [(3.25, 1.25, "C", (2, 0))]},
        {"grids": {"ST1": [[9, 9, 9, 9, 0], [0, 0, 0, 0, 0]],
                   "ST2": [[9, 9, 9, 9, 0], [0, 0, 0, 0, 0]]},
         "cars": [(3.25, 3.4, "C", (2, 0))]},
        {"grids": {"ST1": [[9, 9, 9, 9, 9], [0, 0, 0, 0, 0]],
                   "ST2": [[9, 9, 9, 9, 9], [0, 0, 0, 0, 0]]},
         "cars": []},
    ],
    "Scenario B": [
        {
            # The sheet is already booked: B owns ST2 at t = 2-3, which pushes
            # C onto ST1 at t = 1-2 and A onto ST1 at t = 2-3.
            # Fuel: A 1, B 2, C 2.
            "grids": {"ST1": [[0, B, G, 0, 0], [0, 0, 0, 0, 0]],
                      "ST2": [[0, 0, P, 0, 0], [0, 0, 0, 0, 0]]},
            "cars": [(3.25, 0, "A", (G, 0)),
                     (6.5, 0, "B", (P, P)),
                     (0, 2, "C", (B, B))],
            "time": "0 · B has booked ST2 for t = 2-3, so C takes ST1 first and A waits its turn",
        },
        {
            # A spent its last unit on the cost-1 climb to ST1 and is parked
            # there empty, but the step belongs to C, so its request is refused
            # and recorded in the bottom row.
            # Fuel: A 0, B 1 (one unit into a cost-2 road), C 1.
            "grids": {"ST1": [[X, B, G, 0, 0], [0, G, 0, 0, 0]],
                      "ST2": [[X, 0, P, 0, 0], [0, 0, 0, 0, 0]]},
            "cars": [(3.25, 1.25, "A", (0, 0)),
                     (6.5, 0.85, "B", (P, 0)),
                     (2.8, 2, "C", (B, 0))],
            "time": "1 · C plugs in at ST1 for a full step; A stands at ST1 on an empty tank",
        },
        {
            # C would have charged at ST2 during this step - B holds it, so the
            # refusal is recorded under ST2 and C leaves ST1 with a full tank.
            # Fuel: A 0 (charging), B 0 (charging), C 2.
            "grids": {"ST1": [[X, B, G, 0, 0], [0, G, 0, 0, 0]],
                      "ST2": [[X, X, P, 0, 0], [0, 0, B, 0, 0]]},
            "cars": [(3.25, 1.25, "A", (0, 0)),
                     (6.5, 1.25, "B", (0, 0)),
                     (2.8, 2, "C", (B, B))],
            "time": "2 · A finally charges at ST1 while B charges at ST2; C is refused at ST2",
        },
        {
            # Fuel: A 2 and B 2 (just refilled, leaving on a cost-1 road),
            # C 1 (one unit spent on ST1 -> ST2).
            "grids": {"ST1": [[X, B, G, 0, 0], [0, G, 0, 0, 0]],
                      "ST2": [[X, X, P, 0, 0], [0, 0, B, 0, 0]]},
            "cars": [(3.25, 3.0, "A", (G, G)),
                     (6.5, 3.0, "B", (P, P)),
                     (6.05, 2, "C", (B, 0))],
            "time": "3 · Both sessions end: A and B climb north, C passes ST2 without stopping",
        },
        {
            # Fuel: A 1 and B 1 (a unit spent on the climb), C 0 (its last unit
            # spent leaving the corridor).
            "grids": {"ST1": [[X, B, G, X, 0], [0, G, 0, 0, 0]],
                      "ST2": [[X, X, P, X, 0], [0, 0, B, 0, 0]]},
            "cars": [(3.25, 3.75, "A", (G, 0)),
                     (6.5, 3.75, "B", (P, 0)),
                     (8.0, 2, "C", (0, 0))],
            "time": "4 · A reaches its goal a step late; C leaves the corridor a step late",
        },
        {
            "grids": {"ST1": [[X, B, G, X, X], [0, G, 0, 0, 0]],
                      "ST2": [[X, X, P, X, X], [0, 0, B, 0, 0]]},
            "cars": [(3.25, 3.75, "A", (G, 0)),
                     (6.5, 3.75, "B", (P, 0))],
            "time": "5 · One booking at ST2 propagated through ST1 and delayed two vehicles",
        },
    ],
}

In [43]:
def animate_trace(trace, *, interval=800):
    """Embed a paused, step-by-step player; interval is milliseconds per frame."""
    trace = tuple(trace)
    if not trace:
        raise ValueError("A trace must contain at least one frame.")
    if not math.isfinite(interval) or interval <= 0:
        raise ValueError("interval must be a positive, finite number of milliseconds.")

    fig, ax = plt.subplots(figsize=(11, 6))

    def draw(index):
        state = dict(trace[index])
        time = state.pop("time", index)
        title = f"Step {index + 1}/{len(trace)} · t = {time}"
        plot_step(**state, ax=ax, title=title)

    try:
        animation = FuncAnimation(
            fig, draw, frames=len(trace), interval=interval, repeat=False,
        )
        return HTML(animation.to_jshtml(default_mode="once"))
    finally:
        # The embedded player owns its rendered frames; avoid a duplicate plot.
        plt.close(fig)

## Play a trace

Run the last cell to play `"Scenario A"`. The player
starts paused at step 1 and stops at the final step unless you select **Loop**
or **Reflect**. `interval=800` means 0.8 seconds per step.

For a static snapshot, use `plot_step(**TRACES["Scenario A"][3])`.

In [44]:
animate_trace(TRACES["Scenario B"])